In [1]:
import pandas as pd 
import numpy as np 
import matplotlib as plt
import seaborn as sns
import sqlite3

In [2]:
df = pd.read_csv(r'D:\load_default_project\load_default_project\archive\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv')

C:\Users\gaval\AppData\Local\Temp\ipykernel_3200\1079078025.py:1: DtypeWarning: Columns (0: id, 1: desc, 2: next_pymnt_d, 3: verification_status_joint, 4: sec_app_earliest_cr_line, 5: hardship_type, 6: hardship_reason, 7: hardship_status, 8: hardship_start_date, 9: hardship_end_date, 10: payment_plan_start_date, 11: hardship_loan_status, 12: debt_settlement_flag_date, 13: settlement_status, 14: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'D:\load_default_project\load_default_project\archive\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv')


In [3]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.shape

(2260701, 151)

In [5]:
conn = sqlite3.connect("loan_default_risk.db")

print("SQLite database connected successfully")

SQLite database connected successfully


In [6]:
file_path=(r'D:\load_default_project\load_default_project\archive\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv')

In [7]:
import pandas as pd

columns = [
    'id',
    'loan_amnt',
    'term',
    'int_rate',
    'grade',
    'sub_grade',
    'emp_length',
    'home_ownership',
    'annual_inc',
    'verification_status',
    'issue_d',
    'loan_status',
    'purpose',
    'addr_state',
    'dti',
    'delinq_2yrs',
    'fico_range_low',
    'fico_range_high',
    'inq_last_6mths',
    'revol_util',
    'total_acc',
    'application_type'
]

print("Columns defined:", len(columns))

Columns defined: 22


In [9]:
import pandas as pd

file_path =r'D:\load_default_project\load_default_project\archive\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv'

df = pd.read_csv(
    file_path,
    usecols=columns,
    compression=None,
    low_memory=False
)

print(df.shape)
df.head()

ParserError: Error tokenizing data. C error: out of memory

In [ ]:
df.to_sql(
    "loans_raw",
    conn,
    if_exists="replace",
    index=False
)

print("Dataset loaded into SQLite")

In [ ]:
#Business problem
# Before performing risk analysis, we need to know:
# How many loans exist?
# What columns are available?
# What loan statuses exist?

query = """
SELECT COUNT(*) AS total_records
FROM loans_raw;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Check sample records:
query = """
SELECT *
FROM loans_raw
LIMIT 10;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Check loan statuses:
query = """
SELECT
    loan_status,
    COUNT(*) AS total_loans
FROM loans_raw
GROUP BY loan_status
ORDER BY total_loans DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Before calculating default rates, we need to identify missing values and invalid records.
query = """
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS missing_id,

    SUM(CASE WHEN loan_amnt IS NULL THEN 1 ELSE 0 END) AS missing_loan_amount,

    SUM(CASE WHEN int_rate IS NULL THEN 1 ELSE 0 END) AS missing_interest_rate,

    SUM(CASE WHEN grade IS NULL THEN 1 ELSE 0 END) AS missing_grade,

    SUM(CASE WHEN annual_inc IS NULL THEN 1 ELSE 0 END) AS missing_income,

    SUM(CASE WHEN loan_status IS NULL THEN 1 ELSE 0 END) AS missing_status,

    SUM(CASE WHEN dti IS NULL THEN 1 ELSE 0 END) AS missing_dti,

    SUM(CASE WHEN fico_range_low IS NULL THEN 1 ELSE 0 END) AS missing_fico
FROM loans_raw;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Check Duplicate Loans
query = """
SELECT
    id,
    COUNT(*) AS duplicate_count
FROM loans_raw
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#duplicate count
query = """
SELECT
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_records
FROM loans_raw;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Create the Default Flag
query = """
CREATE TABLE loan_analysis AS

SELECT
    *,
    
    CASE
        WHEN loan_status IN ('Charged Off', 'Default')
        THEN 1
        ELSE 0
    END AS default_flag

FROM loans_raw;
"""

conn.execute("DROP TABLE IF EXISTS loan_analysis")
conn.execute(query)
conn.commit()

print("loan_analysis table created")

In [ ]:
query = """
SELECT
    default_flag,
    COUNT(*) AS total_loans
FROM loan_analysis
GROUP BY default_flag;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Calculate Overall Portfolio Risk
#What is the overall size of the loan portfolio and its observed historical default rate?
query = """
SELECT
    COUNT(*) AS total_loans,

    SUM(loan_amnt) AS total_loan_amount,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate,

    ROUND(AVG(loan_amnt), 2) AS average_loan_amount,

    ROUND(AVG(int_rate), 2) AS average_interest_rate

FROM loan_analysis;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Find Default Rate by Loan Grade
# Does loan grade relate to historical default rates?
query = """
SELECT
    grade,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate,

    ROUND(AVG(loan_amnt), 2) AS average_loan_amount,

    ROUND(AVG(int_rate), 2) AS average_interest_rate

FROM loan_analysis

GROUP BY grade

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Customer Segmentation by Income
# Which income segment has the highest observed default rate?
query = """
SELECT
    CASE
        WHEN annual_inc < 40000 THEN 'Low Income'
        WHEN annual_inc < 80000 THEN 'Middle Income'
        WHEN annual_inc < 120000 THEN 'Upper Middle Income'
        ELSE 'High Income'
    END AS income_segment,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE annual_inc IS NOT NULL

GROUP BY income_segment

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Segment Customers by FICO Score
# How does credit score relate to historical default rates?
query = """
SELECT
    CASE
        WHEN fico_range_low < 580 THEN 'Very Low'
        WHEN fico_range_low < 670 THEN 'Low'
        WHEN fico_range_low < 740 THEN 'Good'
        WHEN fico_range_low < 800 THEN 'Very Good'
        ELSE 'Excellent'
    END AS fico_segment,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE fico_range_low IS NOT NULL

GROUP BY fico_segment

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Segment Customers by DTI
# Does higher debt-to-income ratio correspond to higher historical default rates?
query = """
SELECT
    CASE
        WHEN dti < 10 THEN 'Low DTI'
        WHEN dti < 20 THEN 'Moderate DTI'
        WHEN dti < 30 THEN 'High DTI'
        ELSE 'Very High DTI'
    END AS dti_segment,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE dti IS NOT NULL

GROUP BY dti_segment

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Employment Risk
# Does employment length differ in observed default rates?
query = """
SELECT
    emp_length,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE emp_length IS NOT NULL

GROUP BY emp_length

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Home Ownership Risk
query = """
SELECT
    home_ownership,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

GROUP BY home_ownership

ORDER BY default_rate asc;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Loan Purpose Risk
# Which loan purposes have higher observed default rates
query = """
SELECT
    purpose,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

GROUP BY purpose

HAVING COUNT(*) >= 100

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Interest Rate Risk
# Do loans with higher interest rates have higher historical default rates?
query = """
SELECT
    CASE
        WHEN int_rate < 8 THEN '<8%'
        WHEN int_rate < 12 THEN '8%-12%'
        WHEN int_rate < 16 THEN '12%-16%'
        WHEN int_rate < 20 THEN '16%-20%'
        ELSE '20%+'
    END AS interest_rate_band,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE int_rate IS NOT NULL

GROUP BY interest_rate_band

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Delinquency Risk
# Do customers with previous delinquencies show different historical default rates
query = """
SELECT
    CASE
        WHEN delinq_2yrs = 0 THEN 'No Previous Delinquency'
        ELSE 'Previous Delinquency'
    END AS delinquency_segment,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

GROUP BY delinquency_segment

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
#Default Rate by Income Bracket
query = """
SELECT
    CASE
        WHEN annual_inc < 30000 THEN 'Low Income'
        WHEN annual_inc < 60000 THEN 'Middle Income'
        WHEN annual_inc < 100000 THEN 'Upper Middle Income'
        ELSE 'High Income'
    END AS income_bracket,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE annual_inc IS NOT NULL

GROUP BY
    CASE
        WHEN annual_inc < 30000 THEN 'Low Income'
        WHEN annual_inc < 60000 THEN 'Middle Income'
        WHEN annual_inc < 100000 THEN 'Upper Middle Income'
        ELSE 'High Income'
    END

ORDER BY default_rate DESC;
"""

income_result = pd.read_sql_query(query, conn)
income_result

In [ ]:
#Default Rate by Loan Type
query = """
SELECT
    purpose AS loan_type,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE purpose IS NOT NULL

GROUP BY purpose

ORDER BY default_rate DESC;
"""

loan_type_result = pd.read_sql_query(query, conn)
loan_type_result

In [ ]:
#Default Rate by Geography
query = """
SELECT
    addr_state AS state,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE addr_state IS NOT NULL

GROUP BY addr_state

ORDER BY default_rate DESC;
"""

geography_result = pd.read_sql_query(query, conn)
geography_result

In [ ]:
#Revolving Credit Utilization
# Does revolving utilization relate to historical default rates?
query = """
SELECT
    CASE
        WHEN revol_util < 30 THEN 'Low Utilization'
        WHEN revol_util < 60 THEN 'Moderate Utilization'
        WHEN revol_util < 80 THEN 'High Utilization'
        ELSE 'Very High Utilization'
    END AS utilization_segment,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE revol_util IS NOT NULL

GROUP BY utilization_segment

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

Identify High-Risk Applications

In [ ]:
# Create Risk Score
query = """
CREATE TABLE loan_risk AS

SELECT
    *,
    (
        CASE
            WHEN fico_range_low < 650 THEN 2
            ELSE 0
        END
        +
        CASE
            WHEN dti > 30 THEN 2
            ELSE 0
        END
        +
        CASE
            WHEN grade IN ('E', 'F', 'G') THEN 2
            ELSE 0
        END
        +
        CASE
            WHEN int_rate >= 15 THEN 1
            ELSE 0
        END
        +
        CASE
            WHEN delinq_2yrs > 0 THEN 1
            ELSE 0
        END
        +
        CASE
            WHEN revol_util > 80 THEN 1
            ELSE 0
        END

    ) AS risk_score

FROM loan_analysis;
"""

conn.execute("DROP TABLE IF EXISTS loan_risk")
conn.execute(query)
conn.commit()

print("Risk table created")

In [ ]:
# Create Risk Category
query = """
CREATE TABLE loan_risk_final AS

SELECT
    *,

    CASE
        WHEN risk_score >= 5 THEN 'High Risk'
        WHEN risk_score >= 3 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS risk_category,

    CASE
        WHEN risk_score >= 5 THEN 'Manual Review'
        ELSE 'Standard Review'
    END AS review_status

FROM loan_risk;
"""

conn.execute("DROP TABLE IF EXISTS loan_risk_final")
conn.execute(query)
conn.commit()

print("Final risk table created")

In [ ]:
# How Many Applications Need Manual Review?
query = """
SELECT
    review_status,
    COUNT(*) AS total_applications
FROM loan_risk_final
GROUP BY review_status;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Manual Review Queue
query = """
SELECT
    id,
    loan_amnt,
    grade,
    sub_grade,
    int_rate,
    annual_inc,
    fico_range_low,
    dti,
    delinq_2yrs,
    revol_util,
    risk_score,
    risk_category,
    review_status

FROM loan_risk_final

WHERE review_status = 'Manual Review'

ORDER BY risk_score DESC,
         fico_range_low ASC,
         dti DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Which Customer Segment Has the Highest Default Rate?
query = """
SELECT
    CASE
        WHEN annual_inc < 40000 THEN 'Low Income'
        WHEN annual_inc < 80000 THEN 'Middle Income'
        WHEN annual_inc < 120000 THEN 'Upper Middle Income'
        ELSE 'High Income'
    END AS customer_segment,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE annual_inc IS NOT NULL

GROUP BY customer_segment

HAVING COUNT(*) >= 100

ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Create Power BI SQL Views
conn.execute("DROP VIEW IF EXISTS vw_portfolio_kpis")

conn.execute("""
CREATE VIEW vw_portfolio_kpis AS

SELECT
    COUNT(*) AS total_loans,
    SUM(loan_amnt) AS total_loan_amount,
    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate,

    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate

FROM loan_analysis;
""")

conn.commit()

In [ ]:
pd.read_sql_query(
    "SELECT * FROM vw_portfolio_kpis;",
    conn
)

In [ ]:
# Customer Segment View
conn.execute("DROP VIEW IF EXISTS vw_customer_segments")

conn.execute("""
CREATE VIEW vw_customer_segments AS

SELECT
    CASE
        WHEN annual_inc < 40000 THEN 'Low Income'
        WHEN annual_inc < 80000 THEN 'Middle Income'
        WHEN annual_inc < 120000 THEN 'Upper Middle Income'
        ELSE 'High Income'
    END AS customer_segment,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate

FROM loan_analysis

WHERE annual_inc IS NOT NULL

GROUP BY customer_segment;
""")

conn.commit()

In [ ]:
# Risk View
conn.execute("DROP VIEW IF EXISTS vw_risk_analysis")

conn.execute("""
CREATE VIEW vw_risk_analysis AS

SELECT
    risk_category,
    review_status,

    COUNT(*) AS total_applications,

    SUM(default_flag) AS defaulted_loans,

    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate,

    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount

FROM loan_risk_final

GROUP BY
    risk_category,
    review_status;
""")

conn.commit()

In [ ]:
# Manual Review View
conn.execute("DROP VIEW IF EXISTS vw_manual_review")

conn.execute("""
CREATE VIEW vw_manual_review AS

SELECT
    id,
    loan_amnt,
    term,
    int_rate,
    grade,
    sub_grade,
    emp_length,
    home_ownership,
    annual_inc,
    verification_status,
    purpose,
    addr_state,
    dti,
    delinq_2yrs,
    fico_range_low,
    fico_range_high,
    inq_last_6mths,
    revol_util,
    total_acc,
    application_type,
    risk_score,
    risk_category,
    review_status

FROM loan_risk_final

WHERE review_status = 'Manual Review';
""")

conn.commit()

In [ ]:
import pandas as pd

# 1. Portfolio KPIs
pd.read_sql_query(
    "SELECT * FROM vw_portfolio_kpis",
    conn
).to_csv("vw_portfolio_kpis.csv", index=False)

# 2. Customer Segments
pd.read_sql_query(
    "SELECT * FROM vw_customer_segments",
    conn
).to_csv("vw_customer_segments.csv", index=False)

# 3. Risk Analysis
pd.read_sql_query(
    "SELECT * FROM vw_risk_analysis",
    conn
).to_csv("vw_risk_analysis.csv", index=False)

# 4. Manual Review Queue
pd.read_sql_query(
    "SELECT * FROM vw_manual_review",
    conn
).to_csv("vw_manual_review.csv", index=False)

print("Power BI files created successfully.")

In [10]:
conn.execute("DROP VIEW IF EXISTS vw_grade_risk")

conn.execute("""
CREATE VIEW vw_grade_risk AS
SELECT
    grade,
    COUNT(*) AS total_loans,
    SUM(default_flag) AS defaulted_loans,
    ROUND(
        100.0 * SUM(default_flag) / COUNT(*),
        2
    ) AS default_rate
FROM loan_analysis
WHERE grade IS NOT NULL
GROUP BY grade
ORDER BY grade
""")

conn.commit()

print("vw_grade_risk created successfully")

vw_grade_risk created successfully


In [11]:
pd.read_sql_query("""
SELECT *
FROM vw_grade_risk
ORDER BY grade
""", conn)

,grade,total_loans,defaulted_loans,default_rate
0,A,433027,14206,3.28
1,B,663557,52576,7.92
2,C,650053,85657,13.18
3,D,324424,61067,18.82
4,E,135639,36041,26.57
5,F,41800,14492,34.67
6,G,12168,4560,37.48


In [12]:
grade_risk = pd.read_sql_query("""
SELECT *
FROM vw_grade_risk
ORDER BY grade
""", conn)

grade_risk.to_csv(
    "vw_grade_risk.csv",
    index=False
)

print("vw_grade_risk.csv created successfully")

vw_grade_risk.csv created successfully
